Created because there was a bug in computing coherence when I first introduced the Trump dataset

In [ ]:
import polars as pl
import bertopic

import pathlib

DATA = pathlib.Path("../data/processed/trump_embeddings.parquet")

In [ ]:
df = pl.read_parquet(DATA)
df

In [ ]:
def clean_tweet_text(text):
    if not isinstance(text, str):
        return ""
    # Split text into words
    words = text.split()
    # Keep only words that DO NOT start with 'http' and are not 'covfefe'
    clean_words = [w for w in words if not w.startswith('http') and 'covfefe' not in w.lower()]
    return " ".join(clean_words)

In [ ]:
model = bertopic.BERTopic()

docs = df["text"].to_list()
cleaned_docs = [clean_tweet_text(d) for d in docs]
embeddings = df["text_embedding"].to_numpy()

topics, probs = model.fit_transform(documents=cleaned_docs, embeddings=embeddings)

In [ ]:
model = bertopic.BERTopic()

docs = df["clean_text"].to_list()

topics, probs = model.fit_transform(
    documents=df["clean_text"].to_list(),
    embeddings=df["clean_text_embedding"].to_numpy()
)

In [ ]:
import logging
from octis.evaluation_metrics.coherence_metrics import Coherence

def bertopic_output_to_octis(
    m: bertopic.BERTopic,
    topic_assignments: list[int],
    topk: int = 10
) -> dict[str, list[list[str]]]:
    """
    Reshapes BERTopic output so that it can be readily passed to OCTIS
    for evaluation.
    """
    topic_words: list[list[str]] = []
    topic_ids = [
        t_id
        for t_id in m.get_topics().keys() 
        if t_id != -1 # Ignores noise topic
    ]
    for t_id in topic_ids:
        topic_info = m.get_topic(t_id)
        if isinstance(topic_info, list):
            words = [str(word) for word, _ in topic_info[:topk]] # type: ignore
            topic_words.append(words)

    return {"topics": topic_words}

def compute_coherence(
    model_output: dict,
    texts: list[list[str]],
    measure: str = "c_npmi",
    topk: int = 10
) -> float:
    coherence_model = Coherence(
        texts=texts, 
        topk=topk,
        measure=measure
    )
    try:
        return coherence_model.score(model_output)
    except IndexError as e:
        logger = logging.getLogger("pipeline")
        logger.error(f"Error when computing coherence. Model output:\n{model_output["topics"]}")
        raise e
    except ValueError as e:
        logger = logging.getLogger("pipeline")
        logger.error(f"Error when computing coherence. Model output:\n{texts}")
        raise e

In [ ]:
octis_output = bertopic_output_to_octis(model, topics)

analyzer = model.vectorizer_model.build_analyzer()
tokenized_texts = [analyzer(d) for d in docs]
coherence = compute_coherence(octis_output, texts=tokenized_texts)
coherence

In [ ]:
model.get_topic_info()

In [ ]:
docs_s = df["clean_text"]
docs_s.filter(docs_s == "")

In [ ]:
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

def compute_coherence_safe(
    topic_words: list[list[str]], 
    tokenized_texts: list[list[str]], 
    measure: str = "c_npmi", 
    topk: int = 10
) -> float:
    # Build the Gensim Dictionary explicitly
    dictionary = corpora.Dictionary(tokenized_texts)
    
    # CRITICAL STEP: Filter topics to ensure they only contain words present in the dictionary
    # This prevents the "ValueError: unable to interpret topic..."
    valid_topics = []
    for topic in topic_words:
        # Keep word only if it exists in the reference dictionary
        valid_topic = [word for word in topic if word in dictionary.token2id]
        if valid_topic: # Only add non-empty topics
            valid_topics.append(valid_topic)

    # Check if we lost too many words (optional debugging)
    if len(valid_topics) < len(topic_words):
        print(f"Warning: {len(topic_words) - len(valid_topics)} topics were dropped because they contained no valid words in the reference text.")

    # Calculate Coherence using Gensim directly
    coherence_model = CoherenceModel(
        topics=valid_topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=measure,
        topn=topk
    )
    
    return coherence_model.get_coherence()

In [ ]:
compute_coherence_safe(octis_output["topics"], tokenized_texts)

In [ ]:
octis_topics = octis_output["topics"]
octis_topics